### Import the necessary database

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [ ]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [ ]:
#Perfrom land sea mask
land_sea_mask=xr.open_dataset('/work/mh0033/m301036/Data_storage/CMIP6-MPI-ESM-LR/GR15_lsm_regrid.nc')
# land_sea_mask.coords
land_sea_mask

# mask the land area trend data
mask_data = land_sea_mask['var1']
mask_data

In [ ]:
ds_masked = {}
for key in ds.keys():
    ds_masked[key] = ds[key].where(mask_data[0,:,:]==0, drop = False)

In [ ]:
ds_masked_adj = {}
for key in ds_masked.keys():
    ds_masked_adj[key] = preprosess.convert_longitude(ds_masked[key])

In [ ]:
ds_adj = {}
for key in ds.keys():
    ds_adj[key] = preprosess.convert_longitude(ds[key])

In [ ]:
ds_adj

In [ ]:
ds_masked_adj

In [ ]:
lat = ds_masked_adj["ICV_trend_10yr"].lat
lon = ds_masked_adj["ICV_trend_10yr"].lon
# Extratropical South Pacific region
lat1 = 42
lat2 = 60
lon1 = -50
lon2 = -10
def calc_North_Atlantic_anomalies(data,mask_data):
    ds_WH = data.sel(lat=slice(42, 60), lon=slice(-50, -10))
    ds_WH_anomaly = data_process.calc_weighted_mean(ds_WH)
    ds_sel = mask_data.sel(lat=slice(0, 90), lon=slice(-180,180))
    ds_sel_anomaly = data_process.calc_weighted_mean(ds_sel)
    
    ds_anomalies = ds_WH_anomaly - ds_sel_anomaly
    return ds_anomalies

In [ ]:
ds_subpolar_gyre_anom = {}
for key in ds_masked_adj.keys():
    ds_subpolar_gyre_anom[key] = calc_North_Atlantic_anomalies(ds_adj[key], ds_masked_adj[key])

In [ ]:
ds_subpolar_gyre_anom

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [ ]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [ ]:
# 5%---[0]
unforced_trend_subpolar_gyre_lower_percentile = {}

# 95%---[1]
unforced_trend_subpolar_gyre_upper_percentile = {}

for key in ds_subpolar_gyre_anom.keys():
    unforced_trend_subpolar_gyre_lower_percentile[key], unforced_trend_subpolar_gyre_upper_percentile[key] = calc_percentile(ds_subpolar_gyre_anom[key], 5)
    

In [ ]:
unforced_trend_subpolar_gyre_lower_percentile

In [ ]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_subpolar_gyre_lower_percentile.keys())
unforced_trend_subpolar_gyre_lower_percentile_da = xr.DataArray(
	list(unforced_trend_subpolar_gyre_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_subpolar_gyre_upper_percentile_da = xr.DataArray(
	list(unforced_trend_subpolar_gyre_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [ ]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_subpolar_gyre_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_subpolar_gyre_trend_lower_percentile.nc')
unforced_trend_subpolar_gyre_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_subpolar_gyre_trend_upper_percentile.nc')